# YOLO 3회차 — 로컬 실습 Colab판: 현장 컴퓨터처럼 추론하기

[로컬 실습 가이드](https://github.com/jhjwlee/yolo-training/blob/main/day3/01_yolo_day3_local_guide_v1.0.md)의 스크립트 4개를 **Colab에서 그대로** 실행하는 노트북입니다. 내 PC에 설치가 안 되거나 카메라가 없을 때 같은 내용을 여기서 진행합니다.
웹캠 대신 **영상 파일**을 넣고, 창을 띄우는 대신 결과 영상을 저장해서 봅니다(`--headless` 옵션).

| 단계 | 실행하는 스크립트 | 결과물 |
|---|---|---|
| Step 1 | `01_check_env.py` — 환경 점검 | `check_result.jpg` |
| Step 2 | `02_live_detect.py` — 영상 실시간 탐지와 FPS | `live_detect.mp4` |
| Step 3 | `03_live_zone.py` — 위험구역 규칙 | `live_zone.mp4`, `zone_events.csv` |
| Step 4 | `04_export_benchmark.py` — ONNX·OpenVINO 내보내기, CPU 속도 비교 | `benchmark.csv` |
| Step 5 | (선택) Colab 웹캠으로 사진 한 장 찍어 탐지 | `webcam_photo.jpg` |

**실행 환경**: Google Colab, 런타임 유형 **T4 GPU**. Step 4는 일부러 **CPU**로 측정합니다(현장 컴퓨터에는 GPU가 없는 경우가 많기 때문).

## 0. 준비

In [ ]:
# 패키지 설치 — ultralytics(YOLO), lap(추적), onnx·onnxruntime(ONNX 내보내기·실행), openvino(인텔 CPU 최적화)
!pip -q install ultralytics lap onnx onnxruntime openvino

In [ ]:
import os, json, shutil, base64, subprocess, time
from pathlib import Path

# 영상·이미지 처리(cv2), 배열(numpy), 표(pandas), 그래프(matplotlib)
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, display  # 셀 출력에 HTML(영상 플레이어 등) 띄우기
from ultralytics import YOLO

print('GPU 사용 가능:', torch.cuda.is_available())  # T4 런타임이면 True

try:
    from google.colab import drive  # Colab에만 있는 모듈 → 성공하면 Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = True  # False면 Drive 대신 런타임 임시 폴더에 저장 (연결이 끊기면 사라짐)
if IN_COLAB and USE_DRIVE:
    drive.mount('/content/drive')
    SAVE_DIR = Path('/content/drive/MyDrive/yolo_day3/edge')  # Drive 안 결과 폴더
else:
    SAVE_DIR = Path('outputs/edge')
SAVE_DIR.mkdir(parents=True, exist_ok=True)  # 중간 폴더까지 생성, 이미 있으면 그냥 통과
print('결과 저장 위치:', SAVE_DIR.resolve())

In [ ]:
# OpenCV(BGR) 이미지를 matplotlib로 보여 주기
def show_bgr(img, title=None, size=(9, 6)):
    plt.figure(figsize=size)
    plt.imshow(img[..., ::-1]); plt.axis('off')  # [..., ::-1]: 색 축을 뒤집어 BGR→RGB, 눈금 숨김
    if title:
        plt.title(title)
    plt.show()

# 여러 장을 격자로 보여 주기 (입력: BGR 이미지 목록, 제목 목록, 한 줄에 놓을 장 수)
def show_grid(imgs, titles=None, cols=3, size=4.5):
    rows = (len(imgs) + cols - 1) // cols  # 필요한 줄 수 (올림)
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    for k, im in enumerate(imgs):
        ax = axes.ravel()[k]
        ax.imshow(im[..., ::-1])
        if titles:
            ax.set_title(titles[k], fontsize=9)
    plt.tight_layout(); plt.show()

# mp4 영상을 노트북 안에서 재생 (입력: 영상 경로, 표시 폭 px)
# OpenCV가 만든 mp4v 영상은 브라우저가 못 틀어서 H.264로 다시 인코딩
def show_video(path, width=640):
    h264 = str(Path(path).with_name(Path(path).stem + '_h264.mp4'))
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(path),
                    '-vcodec', 'libx264', '-pix_fmt', 'yuv420p', h264], check=True)
    data = base64.b64encode(open(h264, 'rb').read()).decode()  # 영상을 글자(base64)로 바꿔 HTML에 직접 넣음
    display(HTML(f'<video width="{width}" controls loop autoplay muted>'
                 f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))

# 파일을 SAVE_DIR로 복사하고 복사된 경로를 돌려줌
def save_out(path):
    dst = SAVE_DIR / Path(path).name
    if Path(path).resolve() != dst.resolve():  # 원본과 대상이 같은 파일이면 복사 생략
        shutil.copy(path, dst)
    return dst

In [ ]:
# 로컬 실습 스크립트를 GitHub에서 내려받기 (PC에서 쓰는 것과 같은 파일)
BASE = 'https://raw.githubusercontent.com/jhjwlee/yolo-training/main/day3/local'
SCRIPTS = ['01_check_env.py', '02_live_detect.py', '03_live_zone.py', '04_export_benchmark.py']
for s in SCRIPTS:
    if not Path(s).exists():
        !wget -q {BASE}/{s} -O {s}
# 실습 영상: 2회차 02에서 쓴 OpenCV 공개 샘플 (건물 앞 광장 고정 카메라)
VIDEO = 'vtest.avi'
if not Path(VIDEO).exists():
    !wget -q https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi -O {VIDEO}
print(sorted(p.name for p in Path('.').glob('0*.py')), Path(VIDEO).exists())

---
## Step 1. 환경 점검 — `01_check_env.py`

Colab에는 카메라가 없으므로 `--no-camera`로 실행합니다. 스크립트가 버전과 장치를 확인하고 예시 사진으로 YOLO를 한 번 돌립니다.
`!python 파일.py`는 노트북 셀에서 터미널 명령을 실행하는 방법입니다(PC 터미널에서 `python 파일.py`와 같음).

In [ ]:
!python 01_check_env.py --no-camera
show_bgr(cv2.imread('check_result.jpg'), 'check_result.jpg', size=(6, 7))

`[3] 사용할 장치`가 `cuda`로 나오면 T4 GPU를 쓰는 것입니다. 내 노트북에서 실행하면 보통 `cpu`(또는 맥의 `mps`)로 나옵니다.

---
## Step 2. 실시간 탐지 — `02_live_detect.py`

웹캠 대신 영상 파일을 넣고, 결과를 `live_detect.mp4`로 저장합니다. 화면 왼쪽 위 FPS(초당 처리 프레임 수)를 보세요.
`--max-frames 300`은 앞 300프레임(30초)만 처리하라는 뜻입니다.

In [ ]:
!python 02_live_detect.py --source {VIDEO} --headless --max-frames 300 --classes 0
save_out('live_detect.mp4')
show_video('live_detect.mp4')

**해 보기** — `--imgsz 320`을 붙여 다시 실행하면 FPS가 얼마나 오르나요? 사람이 작게 찍힌 이 영상에서 놓치는 사람이 늘어나는지도 보세요. 속도와 정확도를 맞바꾸는 대표적인 손잡이입니다.

---
## Step 3. 위험구역 규칙 — `03_live_zone.py`

PC에서는 마우스로 구역을 그리지만, 여기서는 `--zone`으로 꼭짓점을 **화면 비율(0~1)** 로 줍니다. `"0.3,0.55;0.7,0.55;0.7,0.95;0.3,0.95"`는 화면 가운데 아래쪽 사각형입니다.
규칙은 2회차 02 실습과 같습니다: 연속 3프레임 안이면 진입, 연속 5프레임 밖이면 이탈, 3초 넘게 머물면 경고.

In [ ]:
ZONE = '0.3,0.55;0.7,0.55;0.7,0.95;0.3,0.95'  # ✏️ 꼭짓점 '가로,세로;…' (화면 비율)
!python 03_live_zone.py --source {VIDEO} --headless --zone "{ZONE}"
events = pd.read_csv('zone_events.csv')  # 열: event_id, track_id, enter_s, exit_s, dwell_s, warning
save_out('zone_events.csv'); save_out('live_zone.mp4')
display(events.head(10))
print(f'이벤트 {len(events)}건, 경고 {int(events.warning.sum()) if len(events) else 0}건')
show_video('live_zone.mp4')

**해 보기** — 2회차 02 노트북에서 그린 구역 좌표를 픽셀 → 비율(가로 ÷ 768, 세로 ÷ 576)로 바꿔 넣으면 2회차 결과와 비교할 수 있습니다. 스크립트 위쪽의 `DWELL_WARN_S`를 바꾸려면 왼쪽 파일 탭에서 `03_live_zone.py`를 열어 고치고 다시 실행하세요.

---
## Step 4. 내보내기와 CPU 속도 비교 — `04_export_benchmark.py`

같은 모델을 PyTorch(`.pt`), ONNX, OpenVINO 형식으로 **CPU**에서 30번씩 실행해 처리 시간의 중앙값을 비교합니다. Colab CPU는 2코어라 노트북 PC보다 느릴 수 있으니, 절대값보다 **형식 사이의 차이**를 보세요.

In [ ]:
# 입력 크기 640과 320으로 한 번씩 실행. benchmark.csv는 실행할 때마다 새로 쓰이므로 매번 읽어 모아 둠
rows = []
for sz in [640, 320]:
    !python 04_export_benchmark.py --imgsz {sz}
    rows.append(pd.read_csv('benchmark.csv'))
bench = pd.concat(rows, ignore_index=True)  # 열: format, file, size_mb, imgsz, median_ms, fps
bench.to_csv('benchmark_all.csv', index=False); save_out('benchmark_all.csv')
display(bench[['format', 'imgsz', 'size_mb', 'median_ms', 'fps']])
bench.pivot(index='format', columns='imgsz', values='fps').plot.bar(rot=0, figsize=(7, 3.5), title='CPU FPS by format and imgsz')
plt.ylabel('FPS'); plt.grid(alpha=.3, axis='y'); plt.show()

**생각해 보기** — 현장 PC에서 15 FPS가 필요하다면 이 표에서 어떤 조합을 고르겠습니까? 형식만 바꿔도 빨라지는 만큼은 **정확도 손실 없이** 얻는 속도입니다(입력 크기를 줄이는 것과 다름).

### 내 PC로 모델 가져가기

내보낸 파일을 내려받으면 PC의 `02_live_detect.py --model …`에 바로 쓸 수 있습니다.

In [ ]:
DOWNLOAD = False  # ✏️ True면 ONNX 모델을 내 PC로 내려받기
if DOWNLOAD and IN_COLAB:
    from google.colab import files
    files.download('yolo26n.onnx')

---
## Step 5. (선택) Colab 웹캠으로 사진 한 장 탐지

Colab은 브라우저를 통해 내 노트북 카메라로 **사진 한 장**을 찍을 수 있습니다(실시간 영상은 어렵습니다). 아래 셀을 실행하면 카메라 화면과 **Capture** 버튼이 나옵니다. 브라우저가 카메라 권한을 물으면 허용하세요.

In [ ]:
TAKE_PHOTO = False  # ✏️ True로 바꾸고 실행
if TAKE_PHOTO and IN_COLAB:
    from google.colab.output import eval_js
    # 브라우저에서 실행할 자바스크립트: 카메라 화면을 띄우고 버튼을 누르면 사진을 base64 글자로 돌려줌
    js = """
    async function takePhoto() {
      const div = document.createElement('div');
      const btn = document.createElement('button'); btn.textContent = 'Capture'; div.appendChild(btn);
      const video = document.createElement('video'); video.style.display = 'block'; div.appendChild(video);
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div); video.srcObject = stream; await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => btn.onclick = resolve);
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth; canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop(); div.remove();
      return canvas.toDataURL('image/jpeg', 0.9);
    }
    takePhoto();
    """
    data = eval_js(js)  # 'data:image/jpeg;base64,....' 형태의 글자
    Path('webcam_photo.jpg').write_bytes(base64.b64decode(data.split(',')[1]))
    r = YOLO('yolo26n.pt')('webcam_photo.jpg', verbose=False)[0]
    show_bgr(r.plot(), f'{len(r.boxes)} objects')

---
## 정리

- 학습은 GPU가 있는 곳(Colab)에서, 추론은 **카메라 옆 컴퓨터**에서 하는 것이 현장의 기본 구조입니다.
- 같은 스크립트가 웹캠(PC)과 영상 파일(Colab) 모두에서 돌도록 `--source`, `--headless` 옵션으로 입력과 출력을 바꿨습니다.
- 내보내기(ONNX·OpenVINO)는 정확도를 그대로 두고 CPU 속도를 올리는 첫 번째 방법입니다.

## 용어 정리

| 용어 | 뜻 |
|---|---|
| 추론 (inference) | 학습된 모델로 예측만 하는 것입니다. |
| 엣지 장치 | 카메라 옆에서 직접 계산하는 컴퓨터입니다. |
| `!python 파일.py` | 노트북 셀에서 터미널 명령으로 파이썬 스크립트를 실행하는 방법입니다. |
| 명령줄 옵션 | `--source`, `--headless`처럼 스크립트를 실행할 때 뒤에 붙여 동작을 바꾸는 값입니다. |
| headless | 화면 창 없이 실행하는 방식입니다. 결과는 파일로 저장합니다. |
| FPS | 초당 처리한 프레임 수입니다. |
| imgsz | 모델 입력 이미지 한 변 크기(픽셀)입니다. |
| 내보내기 (export) | 모델을 다른 실행 엔진용 형식으로 바꾸는 것입니다. |
| ONNX / OpenVINO | 표준 모델 형식 / 인텔 CPU용 추론 최적화 도구입니다. |
| 중앙값 | 크기순으로 줄 세운 값의 가운데 값입니다. |
| base64 | 그림 같은 파일을 글자로 바꾸는 방식입니다. 브라우저와 파이썬 사이에 사진을 주고받을 때 씁니다. |
| eval_js | Colab에서 브라우저 자바스크립트를 실행하고 결과를 파이썬으로 받는 함수입니다. |